# SECOM 반도체 공정 센서 데이터 - 모델링 및 평가

이 노트북은 `src/preprocess.py`, `src/train.py`, `src/evaluate.py`의 함수를 그대로 재사용하여
불균형 분류 파이프라인 학습 → 평가 → 해석 과정을 대화형으로 재현합니다.

**데이터 누수 방지 원칙**
- 결측치 보정(median imputation), 저분산/상수 변수 제거, 스케일링, SMOTE는 모두 학습(train) 세트에서만 `fit`하고, 검증/테스트 세트에는 `transform`만 적용합니다.
- SMOTE는 train/test 분할 이후, 그리고 각 CV fold 내부에서만 적용됩니다 (`imblearn.pipeline.Pipeline` + `StratifiedKFold` 사용).
- 모델 선택은 Recall 단독이 아니라 PR-AUC, F1, Precision, False Positive 부담을 함께 고려합니다.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, evaluate, preprocess
from src.data_loader import load_raw_secom, get_feature_columns
from src.utils import ensure_dir, save_json, set_random_seed
from src.train import run_model_search, inject_scale_pos_weight, export_spotfire_files

set_random_seed(config.RANDOM_STATE)
for d in (config.DATA_PROCESSED_DIR, config.FIGURES_DIR, config.METRICS_DIR, config.MODELS_DIR):
    ensure_dir(d)
pd.set_option('display.max_columns', 20)

## 1. 데이터 로드 및 Train/Test 분할
01_eda.ipynb에서 확인한 대로, 뚜렷한 시간 추세가 없는 한 기본 평가는 stratified random split을 사용합니다.

In [2]:
df = load_raw_secom(config.DATA_RAW_DIR)
feature_cols = get_feature_columns(df)

time_structure = preprocess.analyze_time_structure(df)
print('time-ordered split 필요 여부 검토:', time_structure)

X_train, X_test, y_train, y_test = preprocess.stratified_split(df, feature_cols)
print(f'train={len(X_train)} (fail rate={y_train.mean():.4f}), test={len(X_test)} (fail rate={y_test.mean():.4f})')

time-ordered split 필요 여부 검토: {'is_monotonic_time': False, 'time_span_days': 337.0, 'fail_rate_by_month': {'2008-01': 0.058823529411764705, '2008-02': 0.05102040816326531, '2008-03': 0.02, '2008-04': 0.061224489795918366, '2008-05': 0.11290322580645161, '2008-06': 0.08955223880597014, '2008-07': 0.14035087719298245, '2008-08': 0.08067940552016985, '2008-09': 0.04116222760290557, '2008-10': 0.04878048780487805, '2008-11': 0.05714285714285714, '2008-12': 0.0}, 'fail_rate_std_across_months': 0.036724103424010716}
train=1253 (fail rate=0.0662), test=314 (fail rate=0.0669)


## 2. 후보 모델 파이프라인 (A/B/C/D)
- **A. Baseline_LogisticRegression**: median imputation + 상수/저분산 변수 제거 + 스케일링, 불균형 처리 없음 (naive 기준선)
- **B. RandomForest_SMOTE**: 동일 전처리 + train fold 내부에서만 SMOTE 적용
- **C. RandomForest_ClassWeight**: 동일 모델, SMOTE 대신 `class_weight='balanced'` 사용 (SMOTE vs class-weight 비교)
- **D. XGBoost_ScalePosWeight**: 결측치 보정 없이 XGBoost의 native missing-value 처리 사용, `scale_pos_weight`로 불균형 보정

In [3]:
model_configs = preprocess.build_model_configs()
pd.DataFrame([{'Model': c.name, 'Preprocessing': c.preprocessing, 'Sampling/Weighting': c.sampling} for c in model_configs])

,Model,Preprocessing,Sampling/Weighting
0,Baseline_LogisticRegression,median_impute+variance_threshold+scale,none
1,RandomForest_SMOTE,median_impute+variance_threshold,SMOTE (train fold only)
2,RandomForest_ClassWeight,median_impute+variance_threshold,class_weight=balanced
3,XGBoost_ScalePosWeight,variance_threshold_only (native missing handli...,scale_pos_weight (from train class ratio)


## 3. RandomizedSearchCV + StratifiedKFold 튜닝
학습 데이터가 작고 불량 클래스가 적으므로, 넓은 GridSearch 대신 작은 탐색 공간의 RandomizedSearchCV를 사용합니다.

In [4]:
fitted_models = {}
cv_summaries = {}
best_params = {}

for cfg in model_configs:
    inject_scale_pos_weight(cfg, y_train)
    print('fitting:', cfg.name)
    best_pipeline, cv_summary, params = run_model_search(cfg, X_train, y_train)
    fitted_models[cfg.name] = best_pipeline
    cv_summaries[cfg.name] = cv_summary
    best_params[cfg.name] = params

pd.DataFrame({name: {m: f"{v['mean']:.3f} ± {v['std']:.3f}" for m, v in summary.items()} for name, summary in cv_summaries.items()}).T

fitting: Baseline_LogisticRegression


/root/.local/lib/python3.11/site-packages/sklearn/model_selection/_search.py:326: UserWarning: The total space of parameters 4 is smaller than n_iter=20. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


fitting: RandomForest_SMOTE


/root/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


/root/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/root/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

/root/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


/root/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


/root/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/root/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


/root/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,recall,precision,f1,pr_auc,roc_auc
Baseline_LogisticRegression,0.120 ± 0.085,0.202 ± 0.123,0.147 ± 0.096,0.161 ± 0.066,0.647 ± 0.062
RandomForest_SMOTE,0.108 ± 0.044,0.305 ± 0.086,0.157 ± 0.056,0.212 ± 0.046,0.695 ± 0.056
RandomForest_ClassWeight,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.214 ± 0.065,0.708 ± 0.059
XGBoost_ScalePosWeight,0.049 ± 0.046,0.283 ± 0.267,0.082 ± 0.078,0.201 ± 0.052,0.687 ± 0.057


## 4. Hold-out 테스트 세트 평가

In [5]:
comparison_rows = []
test_predictions = {}
threshold_tables = {}

for cfg in model_configs:
    pipeline = fitted_models[cfg.name]
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    metrics = evaluate.compute_metrics(y_test.to_numpy(), y_pred, y_proba)
    test_predictions[cfg.name] = (y_test.to_numpy(), y_proba)

    threshold_table = evaluate.threshold_tuning_table(y_test.to_numpy(), y_proba)
    threshold_tables[cfg.name] = threshold_table
    threshold_table.to_csv(config.METRICS_DIR / f'threshold_tuning_{cfg.name}.csv', index=False)
    evaluate.plot_threshold_curve(threshold_table, cfg.name)
    evaluate.plot_confusion_matrix(y_test.to_numpy(), y_pred, cfg.name)

    comparison_rows.append({
        'Model': cfg.name, 'Preprocessing': cfg.preprocessing, 'Sampling_Weighting': cfg.sampling,
        'Recall': metrics['recall'], 'Precision': metrics['precision'], 'F1': metrics['f1'],
        'PR_AUC': metrics['pr_auc'], 'ROC_AUC': metrics['roc_auc'],
        'False_Positives': metrics['false_positives'], 'False_Negatives': metrics['false_negatives'],
        'CV_PR_AUC_mean': cv_summaries[cfg.name]['pr_auc']['mean'], 'CV_PR_AUC_std': cv_summaries[cfg.name]['pr_auc']['std'],
        'CV_Recall_mean': cv_summaries[cfg.name]['recall']['mean'], 'CV_Recall_std': cv_summaries[cfg.name]['recall']['std'],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(config.METRICS_DIR / 'model_comparison.csv', index=False)
comparison_df

,Model,Preprocessing,Sampling_Weighting,Recall,Precision,F1,PR_AUC,ROC_AUC,False_Positives,False_Negatives,CV_PR_AUC_mean,CV_PR_AUC_std,CV_Recall_mean,CV_Recall_std
0,Baseline_LogisticRegression,median_impute+variance_threshold+scale,none,0.142857,0.272727,0.187500,0.129451,0.639525,8,18,0.160629,0.065661,0.119853,0.084849
1,RandomForest_SMOTE,median_impute+variance_threshold,SMOTE (train fold only),0.047619,0.166667,0.074074,0.230793,0.829189,5,20,0.212423,0.045704,0.108088,0.043724
2,RandomForest_ClassWeight,median_impute+variance_threshold,class_weight=balanced,0.000000,0.000000,0.000000,0.219923,0.784983,1,21,0.214340,0.065347,0.000000,0.000000
3,XGBoost_ScalePosWeight,variance_threshold_only (native missing handli...,scale_pos_weight (from train class ratio),0.000000,0.000000,0.000000,0.188505,0.691370,3,21,0.201499,0.051820,0.048529,0.046411


In [6]:
evaluate.plot_pr_curve(test_predictions)
evaluate.plot_roc_curve(test_predictions)
plt.show()

## 5. 최종 모델 선택
Recall 단독이 아니라 **테스트 PR-AUC**(불균형 상황에 강건하고 threshold에 무관한 지표)를 1차 기준으로 삼고, F1/Precision/False Positive 부담을 함께 검토합니다.

In [7]:
best_row = comparison_df.sort_values('PR_AUC', ascending=False).iloc[0]
best_model_name = str(best_row['Model'])
best_pipeline = fitted_models[best_model_name]
print('final model:', best_model_name)
best_row

final model: RandomForest_SMOTE


Model                               RandomForest_SMOTE
Preprocessing         median_impute+variance_threshold
Sampling_Weighting             SMOTE (train fold only)
Recall                                        0.047619
Precision                                     0.166667
F1                                            0.074074
PR_AUC                                        0.230793
ROC_AUC                                       0.829189
False_Positives                                      5
False_Negatives                                     20
CV_PR_AUC_mean                                0.212423
CV_PR_AUC_std                                 0.045704
CV_Recall_mean                                0.108088
CV_Recall_std                                 0.043724
Name: 1, dtype: object

## 6. Threshold Tuning 및 비즈니스 목적별 임계값 선택
임계값을 낮추면 Recall(불량 검출률)이 올라가지만 False Positive(오탐, 불필요한 검사비용)가 늘어납니다. 아래 표는 이 트레이드오프를 정량화합니다.

In [8]:
best_threshold_table = threshold_tables[best_model_name]
threshold_recommendations = evaluate.recommend_thresholds(best_threshold_table)
for strategy, row in threshold_recommendations.items():
    print(strategy, '->', {k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})
best_threshold_table

max_f1 -> {'threshold': 0.35, 'recall': 0.619, 'precision': 0.2826, 'f1': 0.3881, 'false_positives': 33.0, 'false_negatives': 8.0, 'true_positives': 13.0, 'true_negatives': 260.0}
high_recall_min_precision_0.2 -> {'threshold': 0.3, 'recall': 0.8095, 'precision': 0.2099, 'f1': 0.3333, 'false_positives': 64.0, 'false_negatives': 4.0, 'true_positives': 17.0, 'true_negatives': 229.0}
high_precision_min_recall_0.5 -> {'threshold': 0.35, 'recall': 0.619, 'precision': 0.2826, 'f1': 0.3881, 'false_positives': 33.0, 'false_negatives': 8.0, 'true_positives': 13.0, 'true_negatives': 260.0}


,threshold,recall,precision,f1,false_positives,false_negatives,true_positives,true_negatives
0,0.05,1.000000,0.067093,0.125749,292,0,21,1
1,0.10,1.000000,0.071672,0.133758,272,0,21,21
2,0.15,1.000000,0.085714,0.157895,224,0,21,69
3,0.20,0.904762,0.107345,0.191919,158,2,19,135
4,0.25,0.857143,0.147541,0.251748,104,3,18,189
5,0.30,0.809524,0.209877,0.333333,64,4,17,229
6,0.35,0.619048,0.282609,0.388060,33,8,13,260
7,0.40,0.238095,0.178571,0.204082,23,16,5,270
8,0.45,0.190476,0.266667,0.222222,11,17,4,282
9,0.50,0.047619,0.166667,0.074074,5,20,1,288


## 7. Feature Importance
**주의**: 아래 중요도는 익명화된 `feature_XXX` ID 기준의 통계적 중요도 순위이며, 실제 물리적 공정/장비의 인과관계로 해석하지 않습니다.

In [9]:
importance_df, importance_method = evaluate.compute_feature_importance(best_pipeline, X_test, y_test, feature_cols)
importance_df.to_csv(config.METRICS_DIR / 'feature_importance.csv', index=False)
evaluate.plot_feature_importance(importance_df, best_model_name, importance_method)
plt.show()
print('importance method:', importance_method)
importance_df.head(20)

importance method: permutation_importance


,feature_name,importance,rank
0,feature_059,0.015656,1
1,feature_247,0.011477,2
2,feature_487,0.011308,3
3,feature_112,0.009246,4
4,feature_385,0.006389,5
5,feature_425,0.005917,6
6,feature_486,0.005860,7
7,feature_028,0.005460,8
8,feature_019,0.004763,9
9,feature_021,0.004025,10


## 8. 최종 메트릭 저장

In [10]:
final_y_true, final_y_proba = test_predictions[best_model_name]
final_metrics_default = evaluate.compute_metrics(final_y_true, (final_y_proba >= 0.5).astype(int), final_y_proba)

final_metrics = {
    'best_model': best_model_name,
    'best_model_preprocessing': best_row['Preprocessing'],
    'best_model_sampling': best_row['Sampling_Weighting'],
    'selection_criterion': (
        'Ranked primarily by held-out test PR-AUC; Recall/Precision/F1/False Positives '
        'reviewed jointly rather than optimizing Recall alone.'
    ),
    'test_metrics_threshold_0.5': final_metrics_default,
    'cross_validation_summary': cv_summaries[best_model_name],
    'threshold_recommendations': threshold_recommendations,
    'importance_method': importance_method,
    'n_train': len(X_train), 'n_test': len(X_test),
    'train_fail_rate': float(y_train.mean()), 'test_fail_rate': float(y_test.mean()),
    'all_models_cv_summary': cv_summaries,
    'all_models_best_params': best_params,
}
save_json(final_metrics, config.METRICS_DIR / 'final_metrics.json')
final_metrics['test_metrics_threshold_0.5']

{'recall': 0.047619047619047616,
 'precision': 0.16666666666666666,
 'f1': 0.07407407407407407,
 'pr_auc': 0.23079269233412836,
 'roc_auc': 0.8291890134893547,
 'accuracy': 0.9203821656050956,
 'confusion_matrix': [[288, 5], [20, 1]],
 'true_negatives': 288,
 'false_positives': 5,
 'false_negatives': 20,
 'true_positives': 1}

## 9. 모델 아티팩트 및 Spotfire 연계 파일 저장

In [11]:
import joblib
for name, pipeline in fitted_models.items():
    joblib.dump(pipeline, config.MODELS_DIR / f'{name}.joblib')
joblib.dump(best_pipeline, config.MODELS_DIR / 'final_model.joblib')

export_spotfire_files(
    df, feature_cols, X_train, X_test, y_train, y_test,
    best_pipeline, best_model_name, importance_df, comparison_df,
)
print('Saved models to outputs/models/, Spotfire CSVs to data/processed/.')

Saved models to outputs/models/, Spotfire CSVs to data/processed/.


---
### 참고
동일한 파이프라인을 스크립트로 한 번에 실행하려면 프로젝트 루트에서 `python src/train.py`를 실행하면 됩니다.